In [ ]:
import pandas as pd
import plotly.express as px
import geopandas as gpd
import plotly.graph_objects as go
import json
import numpy as np 

### Lecture des données

In [ ]:
##### Données du dataframe patients 
df = pd.read_csv("H:/canc_air/data/data_octobre_2023/Pseudonymisation_provisoire_geocoded_spatial.csv", sep = ";")
df.drop('Unnamed: 0', axis=1, inplace=True)


In [ ]:
## Transformation du crs du dataframe WGS84 vers Lambert93 
gdf = (gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x, df.y)).set_crs(epsg=4326))#.to_crs(epsg=2154)
df_iris = gpd.read_file('H:/canc_air/data/zones_geographiques/iris/CONTOURS-IRIS.shp')
df_epci = gpd.read_file('H:/canc_air/data/zones_geographiques/epci/EPCI_SHAPEFILE.shp')
df_dept = gpd.read_file('H:/canc_air/data/zones_geographiques/departements/DEPARTEMENT.shp')


In [ ]:
df_iris.columns

### Réalisation de la jointure et nan jointure 

In [ ]:
print(f"Nombre de patients ayant un code departement nul : {gdf.CODE_DEPT.isnull().sum()}")
gdf_no_na = gdf.dropna(subset=["CODE_DEPT"])

### Mise en forme des données pour réaliser cartographie 

In [ ]:
def mise_en_forme_figure(gdf, df_spatiale, CODE_zone, path_file): 
    ## Nombre de patients par departement : 
    patient_counts = gdf[CODE_zone].value_counts().reset_index()
    patient_counts.columns = [CODE_zone, 'patient_count']

    if CODE_zone == 'CODE_EPCI':
        patient_counts[CODE_zone]= patient_counts[CODE_zone].astype(int)
        df_spatiale[CODE_zone]= df_spatiale[CODE_zone].astype(int)
    else : 
        patient_counts[CODE_zone]= patient_counts[CODE_zone].astype(str)
        df_spatiale[CODE_zone]= df_spatiale[CODE_zone].astype(str)

    ## Ajout des géométries au nouveau tableau des patients par dept : 
    spatial_with_patients = patient_counts.merge(df_spatiale, on=CODE_zone, how='right')

    spatial_with_patients['patient_count'] = spatial_with_patients['patient_count'].fillna(0)

    #spatial_with_patients = spatial_with_patients.dropna(subset=['patient_count'])

    spatial_with_patients_simplified = spatial_with_patients[[CODE_zone, 'patient_count', 'geometry']]

    df_spatiale = df_spatiale.to_crs(epsg=4326)
    # Convert the department shapes to GeoJSON
    df_spatiale.to_file(path_file+".geojson", driver='GeoJSON')

    with open(path_file+".geojson") as f:
        geojson_spatiale = json.load(f)

    return spatial_with_patients_simplified, geojson_spatiale, patient_counts
    

In [ ]:
dept_w_patient, dept_geojson = mise_en_forme_figure(gdf_no_na, df_dept,'CODE_DEPT', "H:/canc_air/data/zones_geographiques/departements/DEPARTEMENT") 

In [ ]:
epci_w_patient, epci_geojson = mise_en_forme_figure(gdf_no_na, df_epci,'CODE_EPCI', "H:/canc_air/data/zones_geographiques/epci/EPCI") 

### Cartographie 

#### Departements

In [ ]:
# Create a custom color scale that corresponds to the quantiles of your data

quantile_list = [0, 0.25, 0.5, 0.75, 1.]
quantiles = departments_with_patients['patient_count'].quantile(quantile_list)

color_scale = [
    [0, 'rgb(255, 237, 160)'],   # color for min to 25th percentile
    [quantiles[0.25]/quantiles[1], 'rgb(254, 217, 118)'],
    [quantiles[0.25]/quantiles[1], 'rgb(254, 178, 76)'],  # color for 25th to 50th percentile
    [quantiles[0.50]/quantiles[1], 'rgb(253, 141, 60)'],
    [quantiles[0.50]/quantiles[1], 'rgb(252, 78, 42)'],   # color for 50th to 75th percentile
    [quantiles[0.75]/quantiles[1], 'rgb(227, 26, 28)'],
    [quantiles[0.75]/quantiles[1], 'rgb(189, 0, 38)'],    # color for 75th to max
    [1.0, 'rgb(128, 0, 38)']
]

##### Autre visualisation 

In [ ]:
contrasted_color_scale = [
    [0.0, '#313695'],  # dark blue
    [0.1, '#4575b4'],  # blue
    [0.2, '#74add1'],  # light blue
    [0.3, '#abd9e9'],  # lighter blue
    [0.4, '#e0f3f8'],  # very light blue
    [0.5, '#fee090'],  # light orange
    [0.6, '#fdae61'],  # orange
    [0.7, '#f46d43'],  # red-orange
    [0.8, '#d73027'],  # red
    [0.9, '#a50026'],  # dark red
    [1.0, '#67001f']   # darker red
]

# Update your figure creation code with the new color scale
fig = px.choropleth_mapbox(dept_w_patient,
                           geojson=dept_geojson,
                           locations="CODE_DEPT",
                           color="patient_count",
                           featureidkey="properties.CODE_DEPT",
                           hover_name="CODE_DEPT",
                           mapbox_style="carto-positron",
                           zoom=5,
                           color_continuous_scale=contrasted_color_scale,
                           center={"lat": 46.2276, "lon": 2.2137},
                           opacity=0.5,
                           labels={'count': 'Nombre de personne'},
                           range_color=(0, 10000)  # Set the range_color to (0, 10000)
                          )

# Update the layout with non-zero margins and a readable colorbar
fig.update_layout(margin={"r":30, "t":30, "l":30, "b":30},  # Updated margins
                  width=1200,
                  height=600,
                  coloraxis_colorbar=dict(
                      title='Patient Count',
                      tickvals=[0, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000],
                      ticktext=['0', '1k', '2k', '3k', '4k', '5k', '6k', '7k', '8k', '9k', '10k']
                  )
)

fig.show()


#### EPCI 

In [ ]:
## Nombre de patients par departement : 
patient_counts = gdf_no_na['CODE_EPCI'].value_counts().reset_index()
patient_counts.columns = ['CODE_EPCI', 'patient_count']

patient_counts['CODE_EPCI']= patient_counts['CODE_EPCI'].astype(int)
df_epci['CODE_EPCI']= df_epci['CODE_EPCI'].astype(int)

## Ajout des géométries au nouveau tableau des patients par dept : 
spatial_with_patients = patient_counts.merge(df_epci, on='CODE_EPCI', how='left')
spatial_with_patients.head()
#spatial_with_patients = spatial_with_patients['patient_count'].fillna(0)

spatial_with_patients_simplified = spatial_with_patients[['CODE_EPCI', 'patient_count', 'geometry']]

df_epci = df_epci.to_crs(epsg=4326)
# Convert the department shapes to GeoJSON
df_epci.to_file("H:/canc_air/data/zones_geographiques/epci/EPCI.geojson", driver='GeoJSON')

with open("H:/canc_air/data/zones_geographiques/epci/EPCI.geojson") as f:
    geojson_epci = json.load(f)

In [ ]:
contrasted_color_scale = [
    [0.0, '#313695'],  # dark blue
    [0.1, '#4575b4'],  # blue
    [0.2, '#74add1'],  # light blue
    [0.3, '#abd9e9'],  # lighter blue
    [0.4, '#e0f3f8'],  # very light blue
    [0.5, '#fee090'],  # light orange
    [0.6, '#fdae61'],  # orange
    [0.7, '#f46d43'],  # red-orange
    [0.8, '#d73027'],  # red
    [0.9, '#a50026'],  # dark red
    [1.0, '#67001f']   # darker red
]

# Update your figure creation code with the new color scale
fig = px.choropleth_mapbox(epci_w_patient,
                           geojson=epci_geojson,
                           locations="CODE_EPCI",
                           color="patient_count",
                           featureidkey="properties.CODE_EPCI",
                           hover_name="CODE_EPCI",
                           mapbox_style="carto-positron",
                           zoom=5,
                           color_continuous_scale=contrasted_color_scale,
                           center={"lat": 46.2276, "lon": 2.2137},
                           opacity=0.5,
                           labels={'count': 'Nombre de personne'},
                           range_color=(0, 10000)  # Set the range_color to (0, 10000)
                          )

# Update the layout with non-zero margins and a readable colorbar
fig.update_layout(margin={"r":30, "t":30, "l":30, "b":30},  # Updated margins
                  width=1200,
                  height=600,
                  coloraxis_colorbar=dict(
                      title='Patient Count',
                      tickvals=[0, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000],
                      ticktext=['0', '1k', '2k', '3k', '4k', '5k', '6k', '7k', '8k', '9k', '10k']
                  )
)

fig.show()

#### IRIS